# Phase 1 — Portfolio, Labelling, and Factor Research

End-to-end walk-through of the Phase 1 hydration:

1. **Structural bars** (dollar / volume / imbalance) via `aqp.data.bars`.
2. **Fractional differentiation** to make series stationary without killing memory.
3. **Triple-barrier + meta-labelling** into a LightGBM classifier.
4. **MDA feature importance** under purged + combinatorial CV.
5. **Portfolio construction** with the new PyPortfolioOpt optimizers (`EfficientCVaR`, `EfficientSemivariance`, `EfficientCDaR`, `CLA`).
6. **Classical forecaster comparison** — `ARIMA`, `Prophet`, `AutoARIMA`, `Naive`.

Dependencies: `pip install "agentic-quant-platform[ml,ml-forecast,portfolio,ml-anomaly]"`.

In [ ]:
import numpy as np
import pandas as pd
from aqp.data.duckdb_engine import DuckDBHistoryProvider

provider = DuckDBHistoryProvider()
tickers = ["AAPL", "MSFT", "NVDA", "SPY", "QQQ", "TSLA", "GOOG", "AMZN"]
bars = provider.get_bars(tickers, start="2020-01-01", end="2024-12-31", interval="1d")
bars = bars.rename(columns={"timestamp": "timestamp"})
bars.head()

## 1. Structural bars

`aqp.data.bars` closes bars on information, not clock time. We can also compute imbalance bars for intraday tick feeds.

In [ ]:
from aqp.data.bars import dollar_bars, volume_bars

aapl = (
    bars[bars["vt_symbol"].str.startswith("AAPL")]
    .set_index("timestamp")[["close", "volume"]]
    .rename(columns={"close": "price"})
)
dbars = dollar_bars(aapl, threshold=1e7)
vbars = volume_bars(aapl, threshold=5e6)
print("daily bars:", len(aapl), "dollar bars:", len(dbars), "volume bars:", len(vbars))
dbars.tail()

## 2. Fractional differentiation

`find_min_d` returns the smallest order `d` that makes the log-price series stationary (ADF). We then apply the fixed-width fracdiff transform and feed the output into the model.

In [ ]:
from aqp.data.fracdiff import find_min_d, frac_diff_ffd

log_p = np.log(aapl["price"]).to_frame("log_price")
d = find_min_d(log_p["log_price"])
print("min d =", d)
ffd = frac_diff_ffd(log_p, d=d).dropna()
ffd.tail()

## 3. Triple-barrier + meta-labelling

Use a simple EMA-cross primary signal, attach triple barriers, then meta-label with a LightGBM classifier.

In [ ]:
from aqp.ml.labeling import triple_barrier_labels, meta_labels

close = aapl["price"]
fast = close.ewm(span=8).mean()
slow = close.ewm(span=21).mean()
side = pd.Series(np.where(fast > slow, 1, -1), index=close.index)
triggers = close.index[::5]  # every 5 bars

labels = triple_barrier_labels(close=close, triggers=triggers, pt_sl=(2.0, 1.5), num_days=5, side=side.loc[triggers])
forward_ret = close.pct_change(5).shift(-5).reindex(labels.index)
meta = meta_labels(primary_side=side.loc[labels.index], forward_returns=forward_ret)
labels.join(meta, how="inner").head()

## 4. MDA feature importance under CPCV

Use `CombinatorialPurgedCV` for stability and permute each feature to rank them. Great diagnostic to add to any new alpha.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from aqp.data.cv import CombinatorialPurgedCV
from aqp.ml.importance import mda_importance

features = pd.DataFrame({
    "ret_1d": close.pct_change().shift(1),
    "ret_5d": close.pct_change(5).shift(1),
    "vol_20d": close.pct_change().rolling(20).std().shift(1),
    "ema_diff": (fast - slow).shift(1),
    "ffd": ffd["log_price"].shift(1),
}).dropna()

aligned = features.reindex(meta.index).dropna()
y = meta.reindex(aligned.index).astype(int)
aligned = aligned.reset_index().rename(columns={"index": "timestamp"}).set_index("timestamp")
cpcv = CombinatorialPurgedCV(n_splits=6, n_test_splits=2, embargo_days=1, date_column="timestamp")
imp = mda_importance(
    fit_model_factory=lambda: RandomForestClassifier(n_estimators=200, n_jobs=-1),
    X=aligned.reset_index(),
    y=y.reset_index(drop=True),
    splitter=cpcv,
    scoring="accuracy",
    predict_proba=True,
)
imp

## 5. Portfolio optimisation — CVaR vs Semivariance vs CLA

Bolt the new optimizers onto a simple momentum alpha to see the tail-risk profile change.

In [ ]:
from aqp.strategies.portfolio_opt import (
    EfficientCVaRPortfolio,
    EfficientSemivariancePortfolio,
    EfficientCDaRPortfolio,
    CLAPortfolio,
    DiscreteAllocation,
)
from aqp.core.types import Direction, Signal, Symbol


def momentum_signals(bars_df: pd.DataFrame, lookback: int = 63, k: int = 8) -> list[Signal]:
    pivot = bars_df.pivot(index="timestamp", columns="vt_symbol", values="close").sort_index()
    mom = pivot.pct_change(lookback).iloc[-1].sort_values(ascending=False)
    top = mom.head(k)
    return [
        Signal(
            symbol=Symbol.parse(vt),
            strength=float(max(0.0, v)),
            direction=Direction.LONG,
            confidence=1.0,
            horizon_days=21,
            source="momentum",
        )
        for vt, v in top.items()
    ]


signals = momentum_signals(bars)
context = {"history": bars}

for cls in (EfficientCVaRPortfolio, EfficientSemivariancePortfolio, EfficientCDaRPortfolio, CLAPortfolio):
    pcm = cls(max_positions=8, lookback_bars=252)
    targets = pcm.construct(signals, context)
    print(f"\n{cls.__name__} targets:")
    for t in targets:
        print(f"  {t.symbol}: {t.target_weight:+.3f}  [{t.rationale}]")

### Discrete allocation

Round the continuous weights into integer shares for $100k of cash.

In [ ]:
pcm = EfficientCVaRPortfolio(max_positions=8)
targets = pcm.construct(signals, context)

weights = {t.symbol.vt_symbol: t.target_weight for t in targets}
prices = (
    bars[bars["vt_symbol"].isin(weights)]
    .sort_values("timestamp")
    .groupby("vt_symbol")["close"]
    .last()
    .to_dict()
)
result = DiscreteAllocation(method="greedy").allocate(
    weights=weights,
    latest_prices=prices,
    total_portfolio_value=100_000.0,
)
print("shares:", result.shares)
print("leftover cash:", result.leftover_cash)

## 6. Classical forecasters

Plug `NaiveForecaster`, `ARIMAForecaster`, `ProphetForecaster`, and `AutoARIMAForecaster` behind the same `BaseForecaster` interface.

In [ ]:
from aqp.ml.applications.forecaster import NaiveForecaster
try:
    from aqp.ml.applications.forecaster import ARIMAForecaster, ProphetForecaster, AutoARIMAForecaster
    forecasters = {
        "Naive": NaiveForecaster(strategy="last"),
        "ARIMA": ARIMAForecaster(order=(2, 1, 2)),
        "Prophet": ProphetForecaster(weekly_seasonality=True),
        "AutoARIMA": AutoARIMAForecaster(seasonal=False, max_p=5, max_q=5),
    }
except Exception as exc:
    print("Install the ml-forecast extra to enable statsmodels / prophet / pmdarima", exc)
    forecasters = {"Naive": NaiveForecaster(strategy="last")}

series = aapl["price"].resample("D").last().ffill()
train, test = series.iloc[:-21], series.iloc[-21:]

preds = {}
for name, fc in forecasters.items():
    try:
        fc.fit(train)
        preds[name] = fc.predict(len(test))
    except Exception as exc:
        print(f"{name} failed: {exc}")
pd.concat(preds, axis=1).tail()